In [1]:
transactions = [
    {"id": 1, "amount": 100},
    {"id": 2, "amount": 200},
    {"id": 1, "amount": 100}
]

# Find duplicate transactions by ID
seen_ids = set()
duplicates = []

for transaction in transactions:
    transaction_id = transaction["id"]
    if transaction_id in seen_ids:
        duplicates.append(transaction)
    else:
        seen_ids.add(transaction_id)

duplicates

[{'id': 1, 'amount': 100}]

In [2]:
logs = [
    ("Ahmed", "login"),
    ("Ahmed", "logout"),
    ("John", "login")
]

# Count events per user
counts = {}
for user, event in logs:
    counts[user] = counts.get(user, 0) + 1

counts

{'Ahmed': 2, 'John': 1}

In [10]:
items = ["apple", "banana", "apple", "orange", "banana", "apple"]
counts = {}

for item in items:
    counts[item] = counts.get(item, 0) + 1

counts

{'apple': 3, 'banana': 2, 'orange': 1}

In [11]:
counts.get(item, 0)

3

In [3]:
from collections import Counter

words = ["apple", "banana", "apple", "orange", "banana", "apple"]
word_counts = Counter(words)
word_counts

Counter({'apple': 3, 'banana': 2, 'orange': 1})

In [4]:
nums = [1, 2, 3, 4, 5, 6]
squared_evens = [x * x for x in nums if x % 2 == 0]
squared_evens

[4, 16, 36]

In [5]:
data = [
    {"name": "Book", "price": 15.99},
    {"name": "Pen", "price": 2.50},
    {"name": "Notebook", "price": 5.75}
]
sorted_data = sorted(data, key=lambda x: x["price"])
sorted_data

[{'name': 'Pen', 'price': 2.5},
 {'name': 'Notebook', 'price': 5.75},
 {'name': 'Book', 'price': 15.99}]

In [6]:
from collections import defaultdict

grouped = defaultdict(list)
grouped

defaultdict(list, {})

In [7]:
from collections import defaultdict

products = [
    {"name": "Apple", "category": "Fruit"},
    {"name": "Carrot", "category": "Vegetable"},
    {"name": "Banana", "category": "Fruit"}
]

grouped = defaultdict(list)
for product in products:
    grouped[product["category"]].append(product["name"])

dict(grouped)

{'Fruit': ['Apple', 'Banana'], 'Vegetable': ['Carrot']}

In [8]:
import json

payload = '{"name": "Ahmed", "events": ["login", "logout"]}'
data = json.loads(payload)
data

{'name': 'Ahmed', 'events': ['login', 'logout']}

# The 9 patterns: practice from memory
Run the sample-data cell once. Then fill in each function and run its cell. If the `assert` lines pass, nothing is printed except "✓".

In [ ]:
from collections import Counter, defaultdict, deque
from datetime import datetime, timedelta
import json

rides = [
    {"ride_id": 1, "driver_id": "d1", "city": "Cairo",      "fare": 100.0},
    {"ride_id": 2, "driver_id": "d2", "city": "Almaty",     "fare": 50.0},
    {"ride_id": 3, "driver_id": "d1", "city": "Cairo",      "fare": 80.0},
    {"ride_id": 4, "driver_id": "d3", "city": "Lima",       "fare": 30.0},
    {"ride_id": 5, "driver_id": "d2", "city": "Almaty",     "fare": 70.0},
    {"ride_id": 6, "driver_id": "d1", "city": "Cairo",      "fare": 60.0},
    {"ride_id": 7, "driver_id": "d4", "city": "Alexandria", "fare": 40.0},
]

In [ ]:
# Pattern 1: Count + top-k
# Q: "Top 3 cities by number of rides"
def top_cities(rides, k=3):
    # Counter counts how many times each city appears -> {"Cairo": 3, "Almaty": 2, ...}
    # .most_common(k) sorts by count (highest first) and returns the top k as (city, count) tuples
    return Counter(r["city"] for r in rides).most_common(k)


print(top_cities(rides, 2))          # [('Cairo', 3), ('Almaty', 2)]
assert top_cities(rides, 2) == [("Cairo", 3), ("Almaty", 2)]
assert top_cities([], 3) == []        # empty input -> empty list, no crash
print("✓ pattern 1")

In [ ]:
# Pattern 2: Group + aggregate
# Q: "Average fare per driver"
def avg_fare_per_driver(rides):
    # defaultdict creates [sum, count] automatically the first time we see a driver
    totals = defaultdict(lambda: [0.0, 0])        # driver -> [sum_of_fares, number_of_rides]
    for r in rides:
        t = totals[r["driver_id"]]
        t[0] += r["fare"]                         # add fare to the sum
        t[1] += 1                                 # add 1 to the count
    # one pass done -> divide sum by count for each driver
    return {d: s / c for d, (s, c) in totals.items()}


print(avg_fare_per_driver(rides))    # d1: (100+80+60)/3 = 80
assert avg_fare_per_driver(rides)["d1"] == 80.0
assert avg_fare_per_driver(rides)["d2"] == 60.0
assert avg_fare_per_driver([]) == {}
print("✓ pattern 2")

In [ ]:
# Pattern 3: Dedup, keep the LATEST version of each record
# Q: "The ride status table has many updates per ride. Keep only the newest row per id."
def latest_per_id(events):
    latest = {}                                   # id -> newest event seen so far
    for e in events:
        cur = latest.get(e["id"])                 # None if we haven't seen this id yet
        if cur is None or e["updated_at"] > cur["updated_at"]:
            latest[e["id"]] = e                   # first time OR newer -> replace
    return list(latest.values())


events = [
    {"id": 1, "status": "requested", "updated_at": 1},
    {"id": 1, "status": "completed", "updated_at": 3},
    {"id": 1, "status": "accepted",  "updated_at": 2},   # arrives late, out of order
    {"id": 2, "status": "cancelled", "updated_at": 5},
]
print(latest_per_id(events))
assert {e["id"]: e["status"] for e in latest_per_id(events)} == {1: "completed", 2: "cancelled"}
assert latest_per_id([]) == []
print("✓ pattern 3")

In [ ]:
# Pattern 4: Parse JSON lines safely: count bad records instead of crashing
# Q: "You receive a log file, one JSON per line, some lines are broken. Parse it."
def parse_lines(lines):
    good, bad = [], 0
    for line in lines:
        try:
            rec = json.loads(line)                   # broken JSON -> JSONDecodeError
            if "ride_id" not in rec:
                raise ValueError("missing ride_id")  # valid JSON but missing a required field
            good.append(rec)
        except (json.JSONDecodeError, ValueError):
            bad += 1        # say: "in production I'd send this to a dead-letter queue"
    return good, bad


lines = ['{"ride_id": 1, "fare": 10}',
         '{"ride_id": 2, "fare": ',        # broken JSON
         '{"fare": 5}',                    # missing ride_id
         '{"ride_id": 3, "fare": 7}']
good, bad = parse_lines(lines)
print(good, bad)
assert [r["ride_id"] for r in good] == [1, 3] and bad == 2
assert parse_lines([]) == ([], 0)
print("✓ pattern 4")

In [ ]:
# Pattern 5: Merge overlapping intervals
# Q: "A driver went online/offline many times, intervals overlap. Total online time?"
def merge_intervals(intervals):
    merged = []
    for start, end in sorted(intervals):            # 1) sort by start time
        if merged and start <= merged[-1][1]:        # 2) overlaps the last merged interval?
            merged[-1][1] = max(merged[-1][1], end)  #    -> extend it
        else:
            merged.append([start, end])              # 3) no overlap -> new interval
    return merged


shifts = [(1, 3), (8, 10), (2, 6), (15, 18)]
m = merge_intervals(shifts)
print(m)                                             # [[1, 6], [8, 10], [15, 18]]
print("total online:", sum(e - s for s, e in m))    # 5 + 2 + 3 = 10
assert m == [[1, 6], [8, 10], [15, 18]]
assert merge_intervals([]) == []
print("✓ pattern 5")

In [ ]:
# Pattern 6: Sessionize: a new session starts after a 30-minute gap
# Q: "How many app sessions did this user have?"
def count_sessions(timestamps, gap=timedelta(minutes=30)):
    ts = sorted(timestamps)          # always sort time data first
    if not ts:
        return 0                     # empty input -> 0 sessions
    # zip(ts, ts[1:]) gives neighbour pairs (t1,t2), (t2,t3), ...
    # every gap bigger than 30 min starts a NEW session -> sessions = 1 + number_of_big_gaps
    return 1 + sum(1 for prev, cur in zip(ts, ts[1:]) if cur - prev > gap)


t0 = datetime(2026, 11, 11, 10, 0)
clicks = [t0, t0 + timedelta(minutes=5), t0 + timedelta(minutes=50),   # gap 45 min -> new session
          t0 + timedelta(minutes=60)]
print(count_sessions(clicks))        # 2
assert count_sessions(clicks) == 2
assert count_sessions([]) == 0
assert count_sessions([t0]) == 1
print("✓ pattern 6")

In [ ]:
# Pattern 7: Sliding window: max rides in any 60-minute window
# Q: "What's the busiest hour for this driver/city (any 60 minutes, not clock hours)?"
def max_in_window(timestamps, window=timedelta(hours=1)):
    q, best = deque(), 0             # q holds the timestamps inside the current window
    for t in sorted(timestamps):
        q.append(t)                  # new ride enters the window
        while t - q[0] >= window:    # oldest ride is 60+ min older than t -> drop it
            q.popleft()              # deque.popleft() is O(1), list.pop(0) is O(n)
        best = max(best, len(q))     # window size right now
    return best


t0 = datetime(2026, 11, 11, 10, 0)
times = [t0, t0 + timedelta(minutes=10), t0 + timedelta(minutes=50),
         t0 + timedelta(minutes=70), t0 + timedelta(minutes=200)]
print(max_in_window(times))          # 10:00, 10:10, 10:50 fall inside one hour -> 3
assert max_in_window(times) == 3
assert max_in_window([]) == 0
print("✓ pattern 7")

In [ ]:
# Pattern 8: Process a huge file in batches with a generator (constant memory)
# Q: "The file is 100 GB and doesn't fit in memory. How do you process it?"
def read_in_batches(lines, size=3):
    # In the interview, `lines` would be `open(path)`. A file object yields one line at a time.
    batch = []
    for line in lines:
        batch.append(line)
        if len(batch) == size:
            yield batch              # hand one batch to the caller, PAUSE here, continue on next()
            batch = []
    if batch:
        yield batch                  # don't forget the last, partial batch!


batches = list(read_in_batches(range(7), size=3))
print(batches)                       # [[0, 1, 2], [3, 4, 5], [6]]
assert batches == [[0, 1, 2], [3, 4, 5], [6]]
assert list(read_in_batches([], 3)) == []
print("✓ pattern 8")

In [ ]:
# Pattern 9: Flatten nested JSON  {"a": {"b": 1}} -> {"a.b": 1}
# Q: "Events come as nested JSON. Flatten them into columns for the warehouse."
def flatten(d, parent="", sep="."):
    out = {}
    for k, v in d.items():
        key = f"{parent}{sep}{k}" if parent else k   # build "rider.location.lat"
        if isinstance(v, dict):
            out.update(flatten(v, key, sep))         # nested dict -> recurse, pass the prefix down
        else:
            out[key] = v                             # plain value -> store it
    return out


event = {"ride_id": 1, "rider": {"id": "r9", "location": {"lat": 30.0, "lon": 31.2}}}
print(flatten(event))
assert flatten(event) == {"ride_id": 1, "rider.id": "r9",
                          "rider.location.lat": 30.0, "rider.location.lon": 31.2}
assert flatten({}) == {}
print("✓ pattern 9")